# Library

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Data Loading

In [2]:
df = pd.read_csv('laptop_price.csv', encoding='latin-1')
df

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,1316,Lenovo,Yoga 500-14ISK,2 in 1 Convertible,14.0,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 6500U 2.5GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.8kg,638.00
1299,1317,Lenovo,Yoga 900-13ISK,2 in 1 Convertible,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core i7 6500U 2.5GHz,16GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.3kg,1499.00
1300,1318,Lenovo,IdeaPad 100S-14IBR,Notebook,14.0,1366x768,Intel Celeron Dual Core N3050 1.6GHz,2GB,64GB Flash Storage,Intel HD Graphics,Windows 10,1.5kg,229.00
1301,1319,HP,15-AC110nv (i7-6500U/6GB/1TB/Radeon,Notebook,15.6,1366x768,Intel Core i7 6500U 2.5GHz,6GB,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19kg,764.00


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   Product           1303 non-null   object 
 3   TypeName          1303 non-null   object 
 4   Inches            1303 non-null   float64
 5   ScreenResolution  1303 non-null   object 
 6   Cpu               1303 non-null   object 
 7   Ram               1303 non-null   object 
 8   Memory            1303 non-null   object 
 9   Gpu               1303 non-null   object 
 10  OpSys             1303 non-null   object 
 11  Weight            1303 non-null   object 
 12  Price_euros       1303 non-null   float64
dtypes: float64(2), int64(1), object(10)
memory usage: 132.5+ KB


# Feature Engineering

## Vectorization of categorical columns

#### RAM

In [4]:
df['Ram_GB'] = df['Ram'].str.replace('GB','').astype(int)

#### Weight

In [5]:
df['Weight_kg'] = df['Weight'].str.replace('kg','').astype(float)

#### Storage / Memory

In [6]:
def memory_parse(memory_str):
    total = 0
    parts = memory_str.split('+')

    for part in parts:
        part = part.strip()

        if 'TB' in part:
            num = float(part.split('TB')[0])
            total += num*1024 #Convert ke GB

        elif 'GB' in part:
            num = float(part.split('GB')[0])
            total += num

    return total

df['Storage_GB'] = df['Memory'].apply(memory_parse)

In [7]:
# Create simplified storage type
def get_storage_type(memory):
    if 'SSD' in memory and 'HDD' in memory:
        return 'SSD + HDD'
    elif 'SSD' in memory:
        return 'SSD'
    elif 'HDD' in memory:
        return 'HDD'
    elif 'Flash Storage' in memory:
        return 'Flash Storage'
    elif 'Hybrid' in memory:
        return 'Hybrid'
    else:
        return 'Other Storage'

df['Storage_Type'] = df['Memory'].apply(get_storage_type)

#### Screen size

In [8]:
df['Inches'] = df['Inches'].astype(float)

### GPU

In [9]:
def get_gpu_brand(gpu):
    if 'Nvidia' in gpu :
        return 'Nvidia'
    elif 'AMD' in gpu:
        return 'AMD'
    else:
        return 'Intel'

df['Gpu_Brand'] = df['Gpu'].apply(get_gpu_brand)

In [10]:
def get_gpu_type(gpu_type):
    if any (x in gpu_type for x in ['GTX', 'RTX', 'Quadro']):
        return 'Nvidia_Dedicated'
    elif any (x in gpu_type for x in ['Geforce', 'MX']):
        return 'Nvidia_Entry'
    elif any (x in gpu_type for x in ['Radeon RX', 'FirePro', 'Radeon Pro']):
        return 'AMD_Dedicated'
    elif 'AMD' in gpu_type:
        return 'AMD_Integrated'
    else:
        return 'Intel_Integrated'
    
df['Gpu_Type'] = df['Gpu'].apply(get_gpu_type)

#### Screen Resolution

In [11]:
def get_resolution(res_str):
    if '3840' in res_str:
        return '4K'
    elif '3200' in res_str or '2880' in res_str or '2736' in res_str:
        return 'QHD'
    elif '2560' in res_str or '2304' in res_str or '2400' in res_str or '2256' in res_str or '2160' in res_str:
        return 'FHD+'
    elif '1920' in res_str or '1600' in res_str:
        return 'FHD'
    else:
        return 'HD'
    
df['Resolution'] = df['ScreenResolution'].apply(get_resolution)

#### CPU

In [12]:
def get_cpu_brand(cpu_str):
    if 'Intel' in cpu_str:
        return 'Intel'
    elif 'AMD' in cpu_str:
        return 'AMD'
    else:
        return 'Other'
    
def get_cpu_tier(cpu_str):
    if 'i7' in cpu_str or 'i9' in cpu_str or 'Ryzen 7' in cpu_str or 'Ryzen 9' in cpu_str:
        return 'High'
    elif 'i5' in cpu_str or 'Ryzen 5' in cpu_str:
        return 'Mid'
    elif 'i3' in cpu_str or 'Ryzen 3' in cpu_str:
        return 'Entry'
    else:
        return 'Other'

df['Cpu_Brand'] = df['Cpu'].apply(get_cpu_brand)
df['Cpu_Tier']  = df['Cpu'].apply(get_cpu_tier)

#### Screen (Touchscreen & IPS panel)

In [13]:
df['Touchscreen'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)

In [14]:
df['IPS_Panel'] = df['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)

#### Price 

In [15]:
EURO_TO_IDR = 20344

df['Price_IDR'] = df['Price_euros'] * EURO_TO_IDR

# Create price segment using quartiles
df['Price_Segment'] = pd.qcut(
    df['Price_IDR'],
    q=4,
    labels=['Low', 'Mid-Low', 'Mid-High', 'High']
)

In [16]:
df[['Memory', 'Storage_GB', 'Ram', 'Ram_GB', 'Weight', 'Weight_kg', 'Gpu', 'Gpu_Brand', 'ScreenResolution','Resolution', 'Touchscreen', 'Cpu', 'Cpu_Brand', 'Cpu_Tier']].head(25)

,Memory,Storage_GB,Ram,Ram_GB,Weight,Weight_kg,Gpu,Gpu_Brand,ScreenResolution,Resolution,Touchscreen,Cpu,Cpu_Brand,Cpu_Tier
0,128GB SSD,128.0,8GB,8,1.37kg,1.37,Intel Iris Plus Graphics 640,Intel,IPS Panel Retina Display 2560x1600,FHD+,0,Intel Core i5 2.3GHz,Intel,Mid
1,128GB Flash Storage,128.0,8GB,8,1.34kg,1.34,Intel HD Graphics 6000,Intel,1440x900,HD,0,Intel Core i5 1.8GHz,Intel,Mid
2,256GB SSD,256.0,8GB,8,1.86kg,1.86,Intel HD Graphics 620,Intel,Full HD 1920x1080,FHD,0,Intel Core i5 7200U 2.5GHz,Intel,Mid
3,512GB SSD,512.0,16GB,16,1.83kg,1.83,AMD Radeon Pro 455,AMD,IPS Panel Retina Display 2880x1800,QHD,0,Intel Core i7 2.7GHz,Intel,High
4,256GB SSD,256.0,8GB,8,1.37kg,1.37,Intel Iris Plus Graphics 650,Intel,IPS Panel Retina Display 2560x1600,FHD+,0,Intel Core i5 3.1GHz,Intel,Mid
5,500GB HDD,500.0,4GB,4,2.1kg,2.10,AMD Radeon R5,AMD,1366x768,HD,0,AMD A9-Series 9420 3GHz,AMD,Other
6,256GB Flash Storage,256.0,16GB,16,2.04kg,2.04,Intel Iris Pro Graphics,Intel,IPS Panel Retina Display 2880x1800,QHD,0,Intel Core i7 2.2GHz,Intel,High
7,256GB Flash Storage,256.0,8GB,8,1.34kg,1.34,Intel HD Graphics 6000,Intel,1440x900,HD,0,Intel Core i5 1.8GHz,Intel,Mid
8,512GB SSD,512.0,16GB,16,1.3kg,1.30,Nvidia GeForce MX150,Nvidia,Full HD 1920x1080,FHD,0,Intel Core i7 8550U 1.8GHz,Intel,High
9,256GB SSD,256.0,8GB,8,1.6kg,1.60,Intel UHD Graphics 620,Intel,IPS Panel Full HD 1920x1080,FHD,0,Intel Core i5 8250U 1.6GHz,Intel,Mid


#### Feature Scaling

In [17]:
scaler = MinMaxScaler()

df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price']] = scaler.fit_transform(df[['Ram_GB', 'Storage_GB', 'Weight_kg', 'Price_IDR']])

#### Encode


In [18]:
type_encode = pd.get_dummies(df['TypeName'],prefix='type')
gpu_encode = pd.get_dummies(df['Gpu_Brand'],prefix='gpu')
resolution_encode = pd.get_dummies(df['Resolution'], prefix='res')
cpu_brand_encode = pd.get_dummies(df['Cpu_Brand'],  prefix='cpu')
cpu_tier_encode   = pd.get_dummies(df['Cpu_Tier'],   prefix='tier')

## Vector

In [19]:
laptop_vector = pd.concat([df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price', 'Touchscreen']],
                           type_encode, 
                           gpu_encode,
                           resolution_encode,
                           cpu_brand_encode,
                           cpu_tier_encode
                           ], axis=1)

laptop_vector.index = df['Product']

In [20]:
laptop_vector

,Norm_Ram,Norm_Storage,Norm_Weight,Norm_Price,Touchscreen,type_2 in 1 Convertible,type_Gaming,type_Netbook,type_Notebook,type_Ultrabook,...,res_FHD+,res_HD,res_QHD,cpu_AMD,cpu_Intel,cpu_Other,tier_Entry,tier_High,tier_Mid,tier_Other
Product,,,,,,,,,,,,,,,,,,,,,
MacBook Pro,0.096774,0.047022,0.169576,0.196741,0,False,False,False,False,True,...,True,False,False,False,True,False,False,False,True,False
Macbook Air,0.096774,0.047022,0.162095,0.122353,0,False,False,False,False,True,...,False,True,False,False,True,False,False,False,True,False
250 G6,0.096774,0.097179,0.291771,0.067679,0,False,False,False,True,False,...,False,False,False,False,True,False,False,False,True,False
MacBook Pro,0.225806,0.197492,0.284289,0.398895,0,False,False,False,False,True,...,False,False,True,False,True,False,False,True,False,False
MacBook Pro,0.096774,0.097179,0.169576,0.275038,0,False,False,False,False,True,...,True,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Yoga 500-14ISK,0.032258,0.047022,0.276808,0.078312,1,True,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
Yoga 900-13ISK,0.225806,0.197492,0.152120,0.223629,1,True,False,False,False,False,...,False,False,True,False,True,False,False,True,False,False
IdeaPad 100S-14IBR,0.000000,0.021944,0.201995,0.009283,0,False,False,False,True,False,...,False,True,False,False,True,False,False,False,False,True


### Cosine similarity

In [21]:
def cosine_sim(vect1,vect2):
  norm_1 = np.linalg.norm(vect1)
  norm_2 = np.linalg.norm(vect2)

  cos_sim = (vect1 @ vect2) / (norm_1 * norm_2)
  return cos_sim


### Recommender System

In [22]:
def recsys(type_name, gpu_brand, ram_user, storage_user, weight_user, price_user, resolution, touchscreen, cpu_brand, cpu_tier, top_N=5):
    
    # Input User
    user_input = pd.DataFrame([[ram_user, storage_user, weight_user, price_user]],
                              columns=['Ram_GB', 'Storage_GB', 'Weight_kg', 'Price_IDR'])
    user_norm = scaler.transform(user_input)[0]

    # Query vector

    cossim = pd.Series(0.0,index=laptop_vector.columns)
    cossim[f'type_{type_name}'] = 1.0
    cossim[f'gpu_{gpu_brand}'] = 1.0
    cossim[f'res_{resolution}'] = 1.0    
    cossim[f'cpu_{cpu_brand}'] = 1.0
    cossim[f'tier_{cpu_tier}'] = 1.0
    cossim['Touchscreen']   = touchscreen
    cossim['Norm_Ram']      = user_norm[0]
    cossim['Norm_Storage']  = user_norm[1]
    cossim['Norm_Weight']   = user_norm[2]
    cossim['Norm_Price']    = user_norm[3]

    # Cosine similarity calculation

    score = [cosine_sim(cossim.values, x) for x in laptop_vector.values]
    result = pd.Series(score, index=laptop_vector.index)
    top_result = result.sort_values(ascending=False).head(top_N)

    return top_result

### Model Inference

Mencoba testing reccomender system dengan kriteria
- type_name = Gaming
- gpu_brand = Intel
- ram_gb = 32
- storage_gb = 512
- weight_kg = 1.5
- price_euros = 1200
- resolution= 'FHD',
- touchscreen= 1,
- cpu_brand= 'Intel',
- cpu_tier= 'High'

In [23]:
recsys(
    type_name= 'Gaming',
    gpu_brand= 'Nvidia',
    ram_user= 32,
    storage_user= 512,
    weight_user= 1.5,
    price_user= 1200,
    resolution= 'FHD',
    touchscreen= 1,
    cpu_brand= 'Intel',
    cpu_tier= 'High', 
)

Product
IdeaPad Y700-15ISK    0.981772
Rog G752VL-UH71T      0.954127
Legion Y520-15IKBN    0.903765
Legion Y520-15IKBN    0.902530
Legion Y520-15IKBN    0.902510
dtype: float64

# RECOMMENDER SYSTEM 2 - GENERAL SPECIFICATION


#### Scaling 


In [24]:
df['Norm_Inches'] = scaler.fit_transform(df[['Inches']])

#### Encode

In [25]:
type_encode = pd.get_dummies(df['TypeName'],prefix='type')
gpu_type_encode = pd.get_dummies(df['Gpu_Type'],prefix='gpu_type')
cpu_tier_encode   = pd.get_dummies(df['Cpu_Tier'],   prefix='tier')
storage_encode = pd.get_dummies(df['Storage_Type'], prefix = 'storage')

In [26]:
laptop_vector2 = pd.concat([df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price', 'Norm_Inches', 'IPS_Panel']],
                           type_encode, 
                           gpu_type_encode,
                           cpu_tier_encode,
                           storage_encode
                           ], axis=1)

laptop_vector2.index = df['laptop_ID']

print(laptop_vector2.columns.tolist())


['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price', 'Norm_Inches', 'IPS_Panel', 'type_2 in 1 Convertible', 'type_Gaming', 'type_Netbook', 'type_Notebook', 'type_Ultrabook', 'type_Workstation', 'gpu_type_AMD_Dedicated', 'gpu_type_AMD_Integrated', 'gpu_type_Intel_Integrated', 'gpu_type_Nvidia_Dedicated', 'gpu_type_Nvidia_Entry', 'tier_Entry', 'tier_High', 'tier_Mid', 'tier_Other', 'storage_Flash Storage', 'storage_HDD', 'storage_Hybrid', 'storage_SSD', 'storage_SSD + HDD']


#### Mapping User Segment & Weighted Cosine Similarity

In [27]:
# Profile

weights = {
    'Student / Basic User': { #harga penting, mudah dibawa, 4-8 gb ram cukup
        'Norm_Price'                : 1.0, 
        'Norm_Weight'               : 0.8,  
        'Norm_Ram'                  : 0.5, 
        'Norm_Storage'              : 0.4,
        'Norm_Inches'               : 0.4,
        'IPS_Panel'                 : 0.2,
        'type_Netbook'              : 0.8,
        'type_Notebook'             : 0.7,
        'type_Ultrabook'            : 0.6,
        'gpu_type_AMD_Integrated'   : 0.8,  
        'gpu_type_Intel_Integrated' : 0.8,
        'tier_Entry'                : 0.7,
        'tier_Mid'                  : 0.5,
        'storage_HDD'               : 0.5,
        'storage_SSD'               : 0.6,
    },
    'Office / Business User': { #portable, layar nyaman, cpu mid to high, ssd penting
        'Norm_Ram'                  : 0.9,  
        'Norm_Weight'               : 0.7, 
        'Norm_Price'                : 0.6,
        'Norm_Storage'              : 0.7,
        'Norm_Inches'               : 0.6,
        'IPS_Panel'                 : 0.5,
        'type_Ultrabook'            : 0.9,
        'type_Notebook'             : 0.7,
        'tier_High'                 : 0.9,
        'tier_Mid'                  : 0.7,
        'gpu_type_Intel_Integrated' : 0.6,
        'storage_SSD'               : 1.0,  
    },
    'Gaming User': { #ram besar, layar besar, type gaming, gpu dedicated, cpu kuat, ssd + hdd
        'Norm_Ram'                  : 0.9,  
        'Norm_Storage'              : 0.8,
        'Norm_Inches'               : 0.8, 
        'Norm_Weight'               : 0.3,
        'Norm_Price'                : 0.5,
        'IPS_Panel'                 : 0.5,
        'type_Gaming'               : 1.0,  
        'gpu_type_Nvidia_Dedicated' : 1.0, 
        'gpu_type_AMD_Dedicated'    : 0.8,
        'tier_High'                 : 1.0,  
        'tier_Mid'                  : 0.5,
        'storage_SSD'               : 0.8,
        'storage_SSD + HDD'         : 0.9,  
    },
    'Creative / Design User': { #ram tinggi, layar ips, cpu tinggi untuk processing
        'Norm_Ram'                  : 1.0,  
        'Norm_Storage'              : 0.8,
        'Norm_Inches'               : 0.7,
        'Norm_Weight'               : 0.4,
        'Norm_Price'                : 0.5,
        'IPS_Panel'                 : 1.0,  
        'type_Workstation'          : 0.9,
        'gpu_type_Nvidia_Dedicated' : 1.0,  
        'gpu_type_AMD_Dedicated'    : 0.9,
        'tier_High'                 : 1.0,  
        'storage_SSD'               : 0.9,
    },
    'On-the-Go / Remote Worker': { #ringan, layar kecil
        'Norm_Weight'               : 1.0,  
        'Norm_Inches'               : 0.8, 
        'Norm_Price'                : 0.6,
        'Norm_Ram'                  : 0.5,
        'Norm_Storage'              : 0.4,
        'IPS_Panel'                 : 0.4,
        'type_Ultrabook'            : 1.0, 
        'type_Netbook'              : 0.8,
        'gpu_type_Intel_Integrated' : 0.6,
        'tier_Mid'                  : 0.7,
        'storage_SSD'               : 0.7,
    },
    'Professional User': { # ram besar, gpu dedicated, cpu tinggi
        'Norm_Ram'                  : 1.0,  
        'Norm_Storage'              : 0.9,
        'Norm_Price'                : 0.5,
        'Norm_Weight'               : 0.4,
        'Norm_Inches'               : 0.6,
        'IPS_Panel'                 : 0.7,
        'type_Workstation'          : 1.0,
        'gpu_type_Nvidia_Dedicated'   : 0.8,  
        'gpu_type_AMD_Dedicated'      : 0.7,
        'tier_High'                 : 1.0,  
        'storage_SSD'               : 0.8,
        'storage_SSD + HDD'         : 0.9,
    },
    'Daily / General User': { # seimbang all around
        'Norm_Ram'                  : 0.7, 
        'Norm_Storage'              : 0.7,
        'Norm_Weight'               : 0.7,
        'Norm_Price'                : 0.8,
        'Norm_Inches'               : 0.6,
        'IPS_Panel'                 : 0.5,
        'type_Notebook'             : 0.9,
        'type_Ultrabook'            : 0.7,
        'gpu_type_Intel_Integrated' : 0.6,
        'gpu_type_AMD_Integrated'   : 0.6,
        'tier_Mid'                  : 0.9, 
        'storage_SSD'               : 0.7,
    }
}

In [28]:
# Weighted Cosine Sim

def weighted_cosine_sim(vect1,vect2,w):
  vect1_w = vect1 *w
  vect2_w = vect2 * w
  norm_1 = np.linalg.norm(vect1_w)
  norm_2 = np.linalg.norm(vect2_w)

  if norm_1 == 0 or norm_2 == 0:
    return 0.0  
  w_cos_sim = (vect1_w @ vect2_w) / (norm_1 * norm_2)
  return w_cos_sim

# Mapping berat & Budget
weight_mapping = {
  'Ringan'  : 0.0,
  'Sedang'  : 0.5,
  'Berat'   : 1.0
}
budget_mapping = {
  'Low'      : 0.0,
  'Mid-Low'  : 0.25,
  'Mid-High' : 0.75,
  'High'     : 1.0
}


In [29]:
# Query default untuk masing masing User need/segment
query_defaults = {
    'Gaming User': {
        'type_Gaming'               : 1.0,
        'gpu_type_Nvidia_Dedicated' : 1.0,
        'tier_High'                 : 1.0,
        'storage_SSD + HDD'         : 1.0,
    },
    'Student / Basic User': {
        'type_Netbook'              : 1.0,
        'gpu_type_Intel_Integrated' : 1.0,
        'tier_Entry'                : 1.0,
        'storage_HDD'               : 1.0,
    },
    'Professional User': {
        'type_Workstation'          : 1.0,
        'gpu_type_Nvidia_Dedicated' : 1.0,
        'tier_High'                 : 1.0,
        'storage_SSD + HDD'         : 1.0,
    },
    'On-the-Go / Remote Worker': {
        'type_Ultrabook'            : 1.0,
        'gpu_type_Intel_Integrated' : 1.0,
        'tier_Mid'                  : 1.0,
        'storage_SSD'               : 1.0,
    },
    'Office / Business User': {
        'type_Ultrabook'            : 1.0,
        'gpu_type_Intel_Integrated' : 1.0,
        'tier_Mid'                  : 1.0,
        'storage_SSD'               : 1.0,
    },
    'Creative / Design User': {
        'type_Workstation'          : 1.0,
        'gpu_type_Nvidia_Dedicated' : 1.0,
        'tier_High'                 : 1.0,
        'storage_SSD'               : 1.0,
        'IPS_Panel'                 : 1.0,
    },
    'Daily / General User': {
        'type_Notebook'             : 1.0,
        'gpu_type_Intel_Integrated' : 1.0,
        'tier_Mid'                  : 1.0,
        'storage_SSD'               : 1.0,
    },
}

In [30]:
def recsys2(user_need, berat, budget, top_N=5):

    # Query vector
    cossim = pd.Series(0.0,index=laptop_vector2.columns)
    cossim['Norm_Weight'] = weight_mapping[berat]
    cossim['Norm_Price'] = budget_mapping[budget]

    # Query default
    for col, val, in query_defaults[user_need].items():
        if col in cossim.index:
            cossim[col] = val

    # Weighted sesuai User Need:
    w = pd.Series(0.1, index=laptop_vector2.columns)
    for col, val in weights[user_need].items():
        if col in w.index:
            w[col] = val

    # Cosine similarity calculation

    score = [weighted_cosine_sim(cossim.values, x, w.values) for x in laptop_vector2.values]
    result = pd.Series(score, index=laptop_vector2.index)
    top_result = result.sort_values(ascending=False).head(top_N)

    result_df = df[df['laptop_ID'].isin(top_result.index)][
        ['Product', 'TypeName', 'Ram', 'Memory', 'Gpu', 'Weight', 'Price_IDR']
    ].copy()
    result_df['Similarity'] = top_result.values
    result_df = result_df.sort_values('Similarity', ascending=False).reset_index(drop=True)

    return result_df

#### Inference

In [31]:
recsys2('Gaming User',berat='Ringan', budget='High')

,Product,TypeName,Ram,Memory,Gpu,Weight,Price_IDR,Similarity
0,Alienware 15,Gaming,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,3.21kg,56447072.72,0.941707
1,GS40 Phantom,Gaming,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 970M,1.7kg,35988536.00,0.932155
2,GS60 Ghost,Gaming,16GB,128GB SSD + 1TB HDD,Nvidia GeForce GTX 970M,1.91kg,43808159.28,0.930414
3,Rog Strix,Gaming,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,2.2kg,49822456.00,0.930208
4,GP62MVR 6RF,Gaming,8GB,128GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,2.2kg,30029981.84,0.930042


In [32]:
recsys2('Student / Basic User',berat='Ringan', budget='Low')

,Product,TypeName,Ram,Memory,Gpu,Weight,Price_IDR,Similarity
0,Vivobook E200HA,Netbook,2GB,32GB Flash Storage,Intel HD Graphics 400,0.98kg,3904013.6,0.796183
1,Chromebook 3,Netbook,4GB,16GB Flash Storage,Intel HD Graphics 400,1.15kg,5472536.0,0.787284
2,VivoBook E12,Netbook,2GB,32GB Flash Storage,Intel HD Graphics 500,1.1kg,4984280.0,0.786240
3,Inspiron 5378,2 in 1 Convertible,4GB,1TB HDD,Intel HD Graphics 620,1.71kg,13203256.0,0.785614
4,Stream 11-Y000na,Netbook,2GB,32GB Flash Storage,Intel HD Graphics 400,1.17kg,4251896.0,0.785531


In [33]:
recsys2('Professional User',berat='Sedang', budget='High')

,Product,TypeName,Ram,Memory,Gpu,Weight,Price_IDR,Similarity
0,Precision 7520,Workstation,16GB,256GB SSD + 1TB HDD,Nvidia Quadro M1200,2.8kg,62150920.00,0.819199
1,Alienware 15,Gaming,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,3.49kg,57242930.00,0.818936
2,ZBook 17,Workstation,8GB,500GB HDD,Nvidia Quadro M1200,3.14kg,38816148.56,0.783725
3,Alienware 15,Gaming,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,3.21kg,56447072.72,0.781623
4,Precision 3520,Workstation,8GB,1TB HDD,Nvidia Quadro M620,2.23kg,36171632.00,0.781587


In [34]:
recsys2('On-the-Go / Remote Worker', berat='Ringan', budget='Mid-Low')

,Product,TypeName,Ram,Memory,Gpu,Weight,Price_IDR,Similarity
0,Pixelbook (Core,Ultrabook,8GB,128GB SSD,Intel HD Graphics 615,1.1kg,25938600.00,0.987653
1,Pixelbook (Core,Ultrabook,8GB,256GB SSD,Intel HD Graphics 615,1.1kg,31716296.00,0.987564
2,EliteBook 820,Ultrabook,4GB,256GB SSD,Intel HD Graphics 520,1.26kg,25307936.00,0.984359
3,Elitebook 820,Ultrabook,4GB,256GB SSD,Intel HD Graphics 620,1.26kg,31716296.00,0.983982
4,Latitude E7270,Ultrabook,8GB,256GB SSD,Intel HD Graphics 520,1.26kg,34856799.28,0.983967
